In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('Preprocessing_V1.csv')

In [ ]:
df.shape

What is the total number of missing or unknown values in the column Gender?

In [ ]:
df['Gender'].isna().sum()


In [ ]:
df['Gender'].value_counts(dropna=False)


In [ ]:
unknown_count = df['Gender'].isin(['Unknown', 'unknown', '?']).sum()
unknown_count


What is the total number of missing or unknown values in the column GlucoseLevel?

In [ ]:
print(df['GlucoseLevel'].unique())

In [ ]:
df['GlucoseLevel'].isna().sum()


In [ ]:
missing_or_unknown = df['GlucoseLevel'].replace(
    ['Unknown', 'unknown', '?', '', 'NA', 'N/A', 'na', 'null', 'NULL'], 
    np.nan
).isna().sum()

print("Total missing or unknown GlucoseLevel values:", missing_or_unknown)

What is the total number of missing or unknown values in the column LivesIn?

In [ ]:
print(df['LivesIn'].unique())

In [ ]:
missing_or_unknown = df['LivesIn'].replace(
    ['Unknown'], 
    np.nan
).isna().sum()

print("Total missing or unknown ", missing_or_unknown)

What is the total number of missing or unknown values in the column BMI?

In [ ]:
print(df['BMI'].unique())

In [ ]:
unknown = df['BMI'].replace(['nan'], np.nan).isna().sum()
print(unknown)

In [ ]:
df['BMI'].isna().sum()

What is the total number of missing or unknown values in the column SmokingStatus?

In [ ]:
print(df['SmokingStatus'].unique())

In [ ]:
value = df['SmokingStatus'].replace(['Unknown'], np.nan).isna().sum()
print(value)

What is the mean value of the BMI in the dataset? Ignore the missing values if any.

In [ ]:
df['BMI'].mean()

How many people live in city, smoked at least once in life and had a heartattack? Ignore records/rows with any missing values.

In [ ]:
print(df['SmokingStatus'].unique())
print(df['HeartAttack'].unique())
print(df['LivesIn'].unique())

In [ ]:
cond = df[(df['LivesIn'] == 'City')&(df['HeartAttack'] == 'Yes')&(df['SmokingStatus'].isin(['smokes', 'formerly smoked']))]
count = cond.shape[0]
print(count)

In [ ]:
df_clean = df.dropna()

# A: female, no tension, no heart disease, never married
A = df_clean[
    (df_clean["Gender"] == "Female") &
    (df_clean["HasTension"] == "No") &
    (df_clean["HeartAttack"] == "No") &
    (df_clean["NeverMarried"] == "Yes")
].shape[0]

# B: female, no tension, no heart disease, married or previously married
B = df_clean[
    (df_clean["Gender"] == "Female") &
    (df_clean["HasTension"] == "No") &
    (df_clean["HeartAttack"] == "No") &
    (df_clean["NeverMarried"] == "No")
].shape[0]

# C: male, no tension, no heart disease, never married
C = df_clean[
    (df_clean["Gender"] == "Male") &
    (df_clean["HasTension"] == "No") &
    (df_clean["HeartAttack"] == "No") &
    (df_clean["NeverMarried"] == "Yes")
].shape[0]

# D: male, tension = yes, heart attack = yes, never married
D = df_clean[
    (df_clean["Gender"] == "Male") &
    (df_clean["HasTension"] == "Yes") &
    (df_clean["HeartAttack"] == "Yes") &
    (df_clean["NeverMarried"] == "Yes")
].shape[0]

A, B, C, D


'HeartAttack' is the target column. What is the distribution count of "No" and "Yes" classes?


In [ ]:
df.HeartAttack.value_counts()

Divide the data into training and test sets
Keep 30% of the data as test set.

Use random_state as 0

HeartAttack is the target, rest of the columns are the features.

For the label/target vector, replace "Yes" with 1 and "No" with 0.

Divide the dataset into training and test sets keeping target(y) in stratified manner.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler, MinMaxScaler
df['HeartAttack'] = df.HeartAttack.map({'Yes': 1, 'No': 0})
X = df.drop('HeartAttack', axis=1)
y = df['HeartAttack']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0) 

In [ ]:
X_train.shape

Prepare a data preprocessing pipeline to process features in following order:
Gender: Impute with most frequent then ordinally encode.
Age: Impute with mean then standard scale.
HasTension: Ordinally encode.
AnyHeartDisease:Ordinally encode.
NeverMarried:Ordinally encode.
Occupation: One hot encode.
LivesIn: Impute with most frequent then ordinally encode.
GlucoseLevel: Impute with mean, then min-max scaling.
BMI: Impute with mean, then standard scale.
SmokingStatus: Impute with most frequent, then one hot encode.
Hint: After transformation, your feature matrix must have columns in following order:

Gender
Age
HasTension
AnyHeartDisease
NeverMarried
Occupation_Govt_job
Occupation_Never_worked
Occupation_Private
Occupation_Self-employed
Occupation_children
LivesIn
GlucoseLevel
BMI
SmokingStatus_formerly smoked
SmokingStatus_never smoked
SmokingStatus_smokes
NOTE:

Make sure to preprocess the features in the above order exactly. Answer(s) of later question(s) depend(s) upon correct order of featuring processing.
You may have to use multiple instances of a trasnformer for this question.

In [ ]:
X_train_modified = X_train.copy()
X_test_modified = X_test.copy()
X_train_modified['SmokingStatus'] = X_train_modified['SmokingStatus'].replace('Unknown', np.nan)
X_test_modified['SmokingStatus'] = X_test_modified['SmokingStatus'].replace('Unknown', np.nan)

In [ ]:
# --- Define Individual Preprocessing Steps ---

# 1. Gender: Impute (most frequent) then Ordinal Encode
gender_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder())
])

# 2. Age: Impute (mean) then Standard Scale
age_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# 3, 4, 5: HasTension, AnyHeartDisease, NeverMarried: Ordinal Encode
tension_pipe = OrdinalEncoder()
heart_pipe = OrdinalEncoder()
married_pipe = OrdinalEncoder()

# 6. Occupation: One hot encode (handle_unknown='ignore' to avoid errors on new categories)
occupation_pipe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# 7. LivesIn: Impute (most frequent) then Ordinal Encode
livesin_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder())
])

# 8. GlucoseLevel: Impute (mean) then Min-Max Scale
glucose_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', MinMaxScaler())
])

# 9. BMI: Impute (mean) then Standard Scale
bmi_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# 10. SmokingStatus: Impute (most frequent) then One hot encode
smoking_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# --- Define the ColumnTransformer ---
# The order of the features below dictates the order of columns in the final matrix.
preprocessor = ColumnTransformer(
    transformers=[
        ('gender_pipe', gender_pipe, ['Gender']),
        ('age_pipe', age_pipe, ['Age']),
        ('tension_pipe', tension_pipe, ['HasTension']),
        ('heart_pipe', heart_pipe, ['AnyHeartDisease']),
        ('married_pipe', married_pipe, ['NeverMarried']),
        ('occupation_pipe', occupation_pipe, ['Occupation']),
        ('livesin_pipe', livesin_pipe, ['LivesIn']),
        ('glucose_pipe', glucose_pipe, ['GlucoseLevel']),
        ('bmi_pipe', bmi_pipe, ['BMI']),
        ('smoking_pipe', smoking_pipe, ['SmokingStatus'])
    ],
    remainder='drop' # Ensures no unintended columns are passed through
)

# Fit and transform the training data
X_train_processed = preprocessor.fit_transform(X_train_modified)

# The shape of the processed training feature matrix is (2800, 16).

Calculate the shape of the feature matrix of training dataset.

In [ ]:
X_train_processed.shape

What is the mean of the transformed test data (features only)?

In [ ]:
# Transform the test set using the *already fitted* preprocessor
X_test_processed = preprocessor.transform(X_test)

# Compute the mean of ALL values in the transformed test matrix
mean_test = X_test_processed.mean()

print("Mean of transformed test data:", mean_test)


If you eliminate 1 feature with recursive feature elimination, which feature will be eliminated?
Type the index of the eliminated feature (index starts from 0).
Use LogisticRegression model with random state as 1729 and rest of the parameters with default values, as an estimator.
Use processed training data.

In [ ]:
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.feature_selection import RFE



In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE

# Logistic Regression model
model = LogisticRegression(random_state=1729)

# RFE: eliminate 1 feature
rfe = RFE(estimator=model, n_features_to_select=X_train.shape[1] - 1)

# Fit RFE
rfe.fit(X_train_processed, y_train)

# Get eliminated feature index (the one marked False)
eliminated_feature_index = list(rfe.support_)

print("Eliminated feature index:", eliminated_feature_index)


In [ ]:
df = pd.read_csv('ModelBuilding_V1.csv')

Load the dataset.

The last column is the target column.
Last 30% rows of the dataset constitute test set and remaining rows form the training set.
Do not shuffle the dataset while splitting
You must have to use only training set to train all the estimator in questions below.
First row of the file has column names/ids, and it has no index column.

Instantiate a perceptron classifier that with following parameters:

random_state = 1729
learning rate = 1
Train for appropriate number of iterations
Do not shuffle the dataset for each iteration.
Include the intercept (bias) term.
Use 10% of the data for validation fraction.
Do not apply regularization.
Set warm start to true.
Hint: one iteration of training indicates going over each sample exactly once.

Train the classifier on the training data.

Q.Train the perceptron classifier for 5 iterations. What is value of bias (intercept) after 5th iteration?

In [ ]:
from sklearn.linear_model import Perceptron
from sklearn.model_selection import train_test_split
X = df.drop('HeartAttack', axis=1)
y = df['HeartAttack']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

In [ ]:
clf = Perceptron(random_state=1729, eta0=1.0, shuffle=False, fit_intercept=True,  validation_fraction=0.1, penalty=None, warm_start=True, max_iter=5)

In [ ]:
clf.fit(X_train.values, y_train)

In [ ]:
clf.intercept_[0]

In continuation of the previous question, compute precision accurate upto 2 decimal places on training data for positive class (i.e. class value 1), after 5 iterations.

In [ ]:
from sklearn.metrics import precision_score

In [ ]:
y_pred=clf.predict(X_train.values)

In [ ]:
precision_score(y_train, y_pred, pos_label=1)

Train (on training data only) logistic regression using SGDClassifier. Use the following parameters:
Choose appropriate loss value to obtain logistic regression
penalty='l2',
eta0=0.001,
alpha=0,
learning_rate='constant'
random_state=1729.
warm_start = True
Train the classifier for 5 iterations and note the value of the loss in each iteration. What will be the loss value after second iteration? Answer upto three decimal places.

Note: Set the remaining parameters, if any, accordingly to be able to get the loss value after second iteration. Also note that the classifier has to be trained for 5 iterations.

In [ ]:
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import log_loss

In [ ]:
clf = SGDClassifier(loss="log_loss", penalty='l2', eta0=0.001, alpha=0, learning_rate="constant", random_state=1729, warm_start=True)

In [ ]:
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import log_loss
clf = SGDClassifier(loss="log_loss", max_iter=1, penalty='l2', eta0=0.001, alpha=0, learning_rate="constant", random_state=1729, warm_start=True)
losses = []
for i in range(5):  
    clf.fit(X_train, y_train)
    y_pred=clf.predict_proba(X_train)                                                                                                                                                                                       
    current_loss = log_loss(y_train, y_pred)
    losses.append(current_loss)

In [ ]:
losses

Use GridSearchCV with SGDClassifier. Following are the classifier's parameters:
loss = 'log_loss'
learning_rate = 'constant'
random_state = 1729
Following are parameters to examine:

alpha = [0.0001, 0.0005, 0.001, 0.005]
eta0 = [0.01, 0.05, 0.1, 0.5]
What are the best values of alpha and eta0 respectively?

In [ ]:
from sklearn.model_selection import GridSearchCV, train_test_split

In [ ]:
sgd = SGDClassifier(loss="log_loss", learning_rate="constant", random_state=1729)
param_grid = {
    'alpha': [0.0001, 0.0005, 0.001, 0.005],
    'eta0': [0.01, 0.05, 0.1, 0.5]
}
grid = GridSearchCV(sgd, param_grid,n_jobs=-1)
grid.fit(X_train, y_train)

In [ ]:
grid.best_params_

Create a new estimator using SGDClassifier that uses the best parameters obtained in Gridsearch earlier ((learning rate to be constant, random_state to be '1729' and use appropriate loss for logistic regression)) and set the weight of class 0 to be 0.1 and the weight of class 1 to be 2. How many samples of class 1 from the test set are correctly predicted by this estimator?

In [ ]:
sgdnew = SGDClassifier(loss="log_loss", learning_rate="constant", random_state=1729, alpha=0.0001, eta0=0.01, class_weight={0:0.1, 1:2} )


In [ ]:
sgdnew.fit(X_train, y_train)

In [ ]:
y_pred=sgdnew.predict(X_test)

In [ ]:
correct_class1 = np.sum((y_test == 1) & (y_pred == 1))
print(correct_class1)

Fit an SVM classifier with following parameters:
kernel='rbf'
decision_function_shape='ovr'
random_state=1729
C=1
Train the model on training data, and print the confusion matrix on test data.

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix

In [ ]:
svm = SVC(kernel='rbf', decision_function_shape='ovr', random_state=1729, C=1)
svm.fit(X_train, y_train)
y_pred=svm.predict(X_test)
cm=confusion_matrix(y_test, y_pred)
print(cm)

Instructions for Q8-10
Train a Decision Tree Classifier with the following properties:

criterion = 'entropy',
splitter = 'random',
min_samples_split = 4,
min_impurity_decrease = 0.0001,
random_state = 1729

Q.8 [MCQ][5 Marks] What is the resultant depth of the tree?
Q.9 [MCQ][Marks 5] How many nodes are there in the tree?
Q.10 [NAT][5 Marks] What is the value of entropy at the left child of root?

In [ ]:
from sklearn.tree import DecisionTreeClassifier


In [ ]:
dt=DecisionTreeClassifier(criterion='entropy', splitter='random', min_samples_split=4, min_impurity_decrease=0.0001, random_state=1729)
dt.fit(X_train, y_train)


In [ ]:
depth=dt.get_depth()
print(depth)

In [ ]:
node = dt.tree_.node_count
print(node)

In [ ]:
tree = dt.tree_
left_child = tree.children_left[0]
entropy_left = tree.impurity[left_child]
print(entropy_left)

Out of DecisionTreeClassifier, KNeighborsClassifier and LogisticRegression, which one performs the best when used as base estimator in BaggingClassifier on the test data in terms of accuracy score when 20 base estimators are used ?
(Use random state 1729 for BaggingClassifier, DecisionTreeClassifier and LogisticRegression)

The metric for best performance will be the lowest 'absolute' difference in the train and test score.

In [ ]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
dt = DecisionTreeClassifier(random_state=1729)
bag1 = BaggingClassifier(dt, n_estimators=20, random_state=1729)
bag1.fit(X_train, y_train)
train_dt = bag1.score(X_train, y_train)
test_dt = bag1.score(X_test, y_test)
result = abs(train_dt-test_dt)
print(result)

In [ ]:
knn = KNeighborsClassifier()
bag2 = BaggingClassifier(knn, n_estimators=20, random_state=1729)
bag2.fit(X_train, y_train)
train_knn = bag2.score(X_train, y_train)
test_knn = bag2.score(X_test, y_test)
result2 = abs(train_knn-test_knn)
print(result2)

In [ ]:
lr = LogisticRegression(random_state=1729)
bag3 = BaggingClassifier(lr, n_estimators=20, random_state=1729)
bag3.fit(X_train, y_train)
train_lr = bag3.score(X_train, y_train)
test_lr = bag3.score(X_test, y_test)
result3 = abs(train_lr-test_lr)
print(result3)

When the above three individual classifiers (with same settings) are used in VotingClassifier, how much absolute difference do we obtain in train and test scores? Enter your answer correct upto 4 decimal places.

In [ ]:
from sklearn.ensemble import VotingClassifier

In [ ]:
dt = DecisionTreeClassifier(random_state=1729)
knn = KNeighborsClassifier()
lr = LogisticRegression(random_state=1729, max_iter=5000)
vclf = VotingClassifier(
    estimators=[
        ('dt', dt),
        ('knn', knn),
        ('lr', lr)
    ]
)

# Train
vclf.fit(X_train, y_train)

# Scores
train_score = vclf.score(X_train, y_train)
test_score = vclf.score(X_test, y_test)

# Absolute difference
abs_diff = abs(train_score - test_score)
print(abs_diff)